# Week 2, Lab 2: Context Window Management - How Your Twin Manages Its Working Memory

## The map before the territory

A model can only look at a limited amount of text on each call. That limit is called its **context window**. It is measured in *tokens*, which are small pieces of text, a bit shorter than a word on average. Everything you send has to fit: the system message, the whole chat history, any document text and the question. The model's own reply needs room too.

In week 1 you kept a `messages` list and sent all of it every time. That works for short chats. A long chat will not fit, so something has to go. This lab builds the part that decides what stays and what goes. It keeps the system message, keeps the newest turns and drops the oldest ones. There is no API call. The Reading tab explains counting size, looping over a list from the end, and building a new list from scratch.

## Exercise

### Goal
Write a function that shrinks a chat history until it fits in a *token budget*. A *token budget* is the most tokens you let yourself send. The system message must always stay, because it says who the twin is. After that, the newest messages matter most, so keep as many of those as fit and drop the oldest.

`count_tokens` is already written. It uses a common rule of thumb: about 4 characters make 1 token. It always gives back at least 1. You only write `trim_messages`.

### 1. `trim_messages(messages, max_tokens)`
Code shape (replace each `...`):
```python
def trim_messages(messages, max_tokens):
    system = messages[:1] if ... else []
    rest = messages[len(system):]
    used = ...
    kept = []
    for message in reversed(rest):
        cost = ...
        if ...:
            break
        kept.append(message)
        used += cost
    kept.reverse()
    return system + kept
```
A *message* is a dict with two keys, `"role"` and `"content"`, like `{"role": "user", "content": "Hi!"}`. A *history* is a list of messages, oldest first.

**You get:** the `messages` list and `max_tokens`, a whole number.
**You give back:** a **new** list. It holds the system message first (if there was one), then the newest messages that fit, still in their original order. The list you were given must not change.

**Idea:**
1. If the first message has the role `"system"`, it goes in `system`. Otherwise `system` is an empty list. `messages[:1]` is a list with just the first message.
2. Everything after it goes in `rest`.
3. Start `used` at the token count of the system message (0 if there is none). Add up `count_tokens(m["content"])` over `system`. The system message is kept even if it is already over the budget.
4. Walk over `rest` from the **newest** to the oldest with `reversed(rest)`. For each message, work out its `cost`. If `used + cost` would go over `max_tokens`, `break`. Otherwise add the message to `kept` and add its cost to `used`.
5. `kept` is now newest-first, so flip it with `kept.reverse()` and give back `system + kept`.

- Stop at the first message that does not fit. Do not skip it and keep going. If you did, the twin would see turn 5 but not turn 4, and the chat would stop making sense.
- If every message fits, you get all of them back.
- If nothing but the system message fits, you get just the system message.

**Check (by hand):** with a system message of 5 tokens and a budget of 12, a newest message of 6 tokens fits (5 + 6 = 11). An older message of 3 tokens does not (11 + 3 = 14), so it is dropped.

### What the asserts below test
The cell below the function builds a small history of four messages and checks four things:
- a big budget keeps all four messages;
- a small budget keeps the system message and only the newest turn;
- the original list still has four messages afterwards (you did not change it);
- with a budget of 1, only the system message is left.

If all four hold, you see `All checks passed.`.

In [ ]:
def count_tokens(text: str) -> int:
    """Rough size of a text: about 4 characters per token, and at least 1."""
    return len(text) // 4 + 1


def trim_messages(messages: list[dict], max_tokens: int) -> list[dict]:
    """Keep the system message and the newest messages that fit in max_tokens.

    You get: a list of {"role": ..., "content": ...} dicts (oldest first) and a token budget.
    You give back: a new list, system message first, then the newest messages that fit,
    in their original order. The list you were given must not change.
    """
    # TODO:
    # 1. system = messages[:1] if the first message has role "system", else [].
    # 2. rest = the messages after the system message.
    # 3. used = the tokens of the system message (use count_tokens on its "content").
    # 4. Walk reversed(rest). If used + cost would be more than max_tokens, break.
    #    Otherwise append the message to kept and add its cost to used.
    # 5. kept.reverse(), then give back system + kept.
    raise NotImplementedError("trim_messages() is not implemented yet - replace the TODO with your code")


history = [
    {"role": "system", "content": "You are Ada's twin."},
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "What do you cook?"},
]
# token sizes: 5, 3, 9, 5

assert trim_messages(history, 100) == history, "a budget of 100 should keep all 4 messages, got " + repr(trim_messages(history, 100))
assert trim_messages(history, 12) == [history[0], history[3]], "a budget of 12 should keep the system message and the newest turn, got " + repr(trim_messages(history, 12))
assert len(history) == 4, "trim_messages must not change the list it was given"
assert trim_messages(history, 1) == [history[0]], "a budget of 1 should keep only the system message, got " + repr(trim_messages(history, 1))

print("All checks passed.")


## Solution

In [ ]:
def count_tokens(text: str) -> int:
    return len(text) // 4 + 1


def trim_messages(messages: list[dict], max_tokens: int) -> list[dict]:
    system = messages[:1] if messages and messages[0]["role"] == "system" else []
    rest = messages[len(system):]
    used = sum(count_tokens(m["content"]) for m in system)
    kept = []
    for message in reversed(rest):
        cost = count_tokens(message["content"])
        if used + cost > max_tokens:
            break
        kept.append(message)
        used += cost
    kept.reverse()
    return system + kept


history = [
    {"role": "system", "content": "You are Ada's twin."},
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "What do you cook?"},
]
print([m["content"] for m in trim_messages(history, 12)])


## Real-world project (offline)

The code cell below builds a longer chat between a user and Ada's twin: a system message and 8 turns. Your project checks how big the whole history is, trims it to a small token budget with your `trim_messages`, and prints what survived. Nothing here calls an LLM. It is the step your code would run before every API call in a long chat.

In [ ]:
"""Load a longer chat history for the twin (embedded so this notebook needs no repo checkout)."""
# lab-data-loader
CHAT_HISTORY = [
    {"role": "system", "content": "You are Ada's digital twin. Answer as Ada, in a warm and short way."},
    {"role": "user", "content": "Hi! Who are you?"},
    {"role": "assistant", "content": "I am Ada's digital twin. Ask me about her work or her weekends."},
    {"role": "user", "content": "What does she do for work?"},
    {"role": "assistant", "content": "She writes Python code and builds web apps every day."},
    {"role": "user", "content": "And what does she do for fun?"},
    {"role": "assistant", "content": "She bakes bread, runs in the park and watches football."},
    {"role": "user", "content": "Has she travelled anywhere lately?"},
    {"role": "assistant", "content": "Last summer she flew to Accra and Lisbon."},
]

print(f"Loaded a chat history of {len(CHAT_HISTORY)} messages.")


In [ ]:
before = sum(count_tokens(m["content"]) for m in CHAT_HISTORY)
trimmed = trim_messages(CHAT_HISTORY, 60)
after = sum(count_tokens(m["content"]) for m in trimmed)

print(f"before: {len(CHAT_HISTORY)} messages, {before} tokens")
print(f"after: {len(trimmed)} messages, {after} tokens")
for message in trimmed:
    print(f"{message['role']}: {message['content']}")
